<a href="https://colab.research.google.com/github/look4pritam/Transformers/blob/master/Notebooks/Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets accelerate evaluate

In [ ]:
!pip install rouge_score

In [ ]:
# ================================================
# Transformer Fine-Tuning for Text Summarization
# Single Script for Google Colab
# ================================================

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate

print("Libraries loaded")

# ==================================================
# 1. Load Dataset
# ==================================================

print("Loading dataset...")

dataset = load_dataset("cnn_dailymail", "3.0.0")

# smaller subset for quick training in Colab
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["validation"].select(range(500))

print("Dataset loaded")


# ==================================================
# 2. Load Transformer Model and Tokenizer
# ==================================================

model_name = "t5-small"

print("Loading model:", model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# ==================================================
# 3. Preprocessing
# ==================================================

max_input_length = 512
max_target_length = 128

def preprocess_function(examples):

    inputs = ["summarize: " + doc for doc in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


print("Tokenizing dataset...")

train_tokenized = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)


# ==================================================
# 4. Data Collator
# ==================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


# ==================================================
# 5. Evaluation Metric
# ==================================================

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return result


# ==================================================
# 6. Training Arguments
# ==================================================

training_args = TrainingArguments(

    output_dir="./t5_summarization",

    learning_rate=2e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    weight_decay=0.01,

    num_train_epochs=2,



    save_strategy="epoch",

    logging_steps=50,


    fp16=torch.cuda.is_available()
)


# ==================================================
# 7. Trainer
# ==================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,


    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# ==================================================
# 8. Train Model
# ==================================================

print("Starting training...")

trainer.train()

In [ ]:
# ==================================================
# 9. Test Summarization
# ==================================================

print("\nTesting summarization...\n")

text = """
Artificial intelligence is transforming the world by enabling machines
to perform tasks that normally require human intelligence. AI systems
are widely used in healthcare, finance, robotics, and natural language
processing.
"""

text = """
United States forces have destroyed 16 Iranian mine-laying boats operating near the Strait of Hormuz, as Washington, DC warned Tehran against attempting to disrupt one of the world's most critical oil shipping routes.
According to the Pentagon, the vessels were targeted as part of efforts to prevent the deployment of naval mines in the strategically vital passage through which a significant portion of global crude oil shipments transits.
The action follows warnings by US President Donald Trump, who said Iran must refrain from placing mines in the waterway or face severe military consequences.
Posting on Truth Social, Trump said the United States wanted any mines that may have been placed in the Strait of Hormuz removed immediately, warning that failure to do so would trigger military consequences 'at a level never seen before'.
In a subsequent post, Trump said US forces had struck and destroyed 10 inactive mine-laying boats or ships, adding that more actions could follow.
The United States Central Command also released a video showing operations aimed at weakening Iran’s ability to project power at sea and threaten international shipping.
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Move inputs to GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:\n", text)
print("\nGenerated Summary:\n", summary)